In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, PolynomialFeatures,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold
from outlier_handler import OutlierHandler 


In [2]:
df=pd.read_csv("F:\DEPI\ML\Pipeline & Feat\sm_risk_data_cleaned (2).csv")
df

,credit_amount,business_age_months,monthly_income_avg,total_deposits_3m,revenue_volatility_3m,request_ratio,dti_monthly,nsf_count_3m,negative_days_3m,owner_percentage,owner_credit_score,p_viable,risk_sharp
0,65044.0008,14,31423.29,86800.05,0.745,2.072,0.542,1,3,77.8,571,0.835,0
1,36058.0700,94,27885.55,79161.32,0.552,1.293,0.434,3,3,84.4,731,0.918,0
2,9782.5900,32,13293.69,42202.89,0.650,0.736,0.432,4,3,81.8,604,0.466,0
3,31340.2500,30,18121.27,51841.87,0.726,1.729,0.306,5,7,53.8,729,0.495,1
4,14167.7000,32,13954.20,42403.16,0.597,1.015,0.311,3,4,66.7,663,0.608,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,39714.2400,65,19329.69,52941.34,0.643,2.055,0.540,2,2,91.4,548,0.727,1
4996,5000.0000,10,5345.41,14075.52,0.730,0.935,0.155,7,2,72.2,630,0.197,1
4997,37128.4300,144,26867.66,77386.24,0.533,1.382,0.666,0,3,87.6,699,0.966,1
4998,26874.4300,30,19612.56,58800.83,0.602,1.370,0.237,1,3,80.4,621,0.820,0


In [3]:
df.describe()

,credit_amount,business_age_months,monthly_income_avg,total_deposits_3m,revenue_volatility_3m,request_ratio,dti_monthly,nsf_count_3m,negative_days_3m,owner_percentage,owner_credit_score,p_viable,risk_sharp
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.00000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,19011.238394,44.470000,15359.883066,46079.320594,0.633891,1.261037,0.347414,3.35720,4.555800,69.325800,660.344600,0.573066,0.467800
std,13676.614881,37.769305,9611.460932,29142.597986,0.082129,0.449257,0.151977,1.84377,2.073776,21.211072,59.317022,0.226339,0.499012
min,5000.000000,4.000000,3598.672600,10774.730800,0.355000,0.400000,0.050000,0.00000,1.000000,10.100000,480.000000,0.040000,0.000000
25%,9069.092500,16.000000,8413.295000,24864.920000,0.579750,0.917000,0.232000,2.00000,3.000000,57.300000,621.000000,0.392000,0.000000
50%,14906.270000,30.000000,12762.030000,38215.210000,0.635000,1.165000,0.344000,3.00000,4.000000,71.800000,661.000000,0.567000,0.000000
75%,24370.097500,62.000000,19506.770000,58613.682500,0.691000,1.557000,0.458000,5.00000,6.000000,86.200000,700.000000,0.755000,1.000000
max,65044.000800,144.000000,46340.388600,140030.183600,0.888000,2.499000,0.787000,8.00000,9.000000,100.000000,830.000000,1.000000,1.000000


In [4]:
df['credit_income_ratio'] = (
    df['credit_amount'] / df['monthly_income_avg']
)
df['deposit_coverage'] = (
    df['total_deposits_3m'] / df['credit_amount']
)
df['revenue_stability'] = (
    df['monthly_income_avg'] /
    (df['revenue_volatility_3m'] + 1)
)
df['nsf_risk_ratio'] = (
    df['nsf_count_3m'] /
    (df['monthly_income_avg'] + 1)
)
df['owner_reliability'] = (
    df['owner_percentage'] *
    df['owner_credit_score']
)
df['request_stress_ratio'] = (
    df['request_ratio'] /
    (df['monthly_income_avg'] + 1)
    
)

In [5]:
df

,credit_amount,business_age_months,monthly_income_avg,total_deposits_3m,revenue_volatility_3m,request_ratio,dti_monthly,nsf_count_3m,negative_days_3m,owner_percentage,owner_credit_score,p_viable,risk_sharp,credit_income_ratio,deposit_coverage,revenue_stability,nsf_risk_ratio,owner_reliability,request_stress_ratio
0,65044.0008,14,31423.29,86800.05,0.745,2.072,0.542,1,3,77.8,571,0.835,0,2.069930,1.334482,18007.616046,0.000032,44423.8,0.000066
1,36058.0700,94,27885.55,79161.32,0.552,1.293,0.434,3,3,84.4,731,0.918,0,1.293074,2.195384,17967.493557,0.000108,61696.4,0.000046
2,9782.5900,32,13293.69,42202.89,0.650,0.736,0.432,4,3,81.8,604,0.466,0,0.735882,4.314081,8056.781818,0.000301,49407.2,0.000055
3,31340.2500,30,18121.27,51841.87,0.726,1.729,0.306,5,7,53.8,729,0.495,1,1.729473,1.654163,10498.997683,0.000276,39220.2,0.000095
4,14167.7000,32,13954.20,42403.16,0.597,1.015,0.311,3,4,66.7,663,0.608,0,1.015300,2.992946,8737.758297,0.000215,44222.1,0.000073
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,39714.2400,65,19329.69,52941.34,0.643,2.055,0.540,2,2,91.4,548,0.727,1,2.054572,1.333057,11764.875228,0.000103,50087.2,0.000106
4996,5000.0000,10,5345.41,14075.52,0.730,0.935,0.155,7,2,72.2,630,0.197,1,0.935382,2.815104,3089.832370,0.001309,45486.0,0.000175
4997,37128.4300,144,26867.66,77386.24,0.533,1.382,0.666,0,3,87.6,699,0.966,1,1.381900,2.084285,17526.196999,0.000000,61232.4,0.000051
4998,26874.4300,30,19612.56,58800.83,0.602,1.370,0.237,1,3,80.4,621,0.820,0,1.370266,2.187984,12242.546816,0.000051,49928.4,0.000070


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   credit_amount          5000 non-null   float64
 1   business_age_months    5000 non-null   int64  
 2   monthly_income_avg     5000 non-null   float64
 3   total_deposits_3m      5000 non-null   float64
 4   revenue_volatility_3m  5000 non-null   float64
 5   request_ratio          5000 non-null   float64
 6   dti_monthly            5000 non-null   float64
 7   nsf_count_3m           5000 non-null   int64  
 8   negative_days_3m       5000 non-null   int64  
 9   owner_percentage       5000 non-null   float64
 10  owner_credit_score     5000 non-null   int64  
 11  p_viable               5000 non-null   float64
 12  risk_sharp             5000 non-null   int64  
 13  credit_income_ratio    5000 non-null   float64
 14  deposit_coverage       5000 non-null   float64
 15  reve

In [7]:
X=df.drop(['p_viable',"risk_sharp"],axis=1)
y=df["risk_sharp"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1,shuffle=True,stratify=y)

In [8]:
print(f"Train features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")
print(f"Train labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")

Train features shape: (4000, 17)
Test features shape: (1000, 17)
Train labels shape: (4000,)
Test labels shape: (1000,)


In [9]:
import pandas as pd
print("on train\n", pd.Series(y_train).value_counts(normalize=True))
print("on test\n", pd.Series(y_test).value_counts(normalize=True))

on train
 risk_sharp
0    0.53225
1    0.46775
Name: proportion, dtype: float64
on test
 risk_sharp
0    0.532
1    0.468
Name: proportion, dtype: float64


In [10]:
num_col=df.select_dtypes(include=np.number).columns
num_col

Index(['credit_amount', 'business_age_months', 'monthly_income_avg',
       'total_deposits_3m', 'revenue_volatility_3m', 'request_ratio',
       'dti_monthly', 'nsf_count_3m', 'negative_days_3m', 'owner_percentage',
       'owner_credit_score', 'p_viable', 'risk_sharp', 'credit_income_ratio',
       'deposit_coverage', 'revenue_stability', 'nsf_risk_ratio',
       'owner_reliability', 'request_stress_ratio'],
      dtype='object')

In [11]:
num_col=['credit_amount', 'business_age_months', 'monthly_income_avg',
       'total_deposits_3m', 'revenue_volatility_3m', 'request_ratio',
       'dti_monthly', 'nsf_count_3m', 'negative_days_3m', 'owner_percentage',
       'owner_credit_score', 'credit_income_ratio',
       'deposit_coverage',  'nsf_risk_ratio',
       'owner_reliability', 'request_stress_ratio']

num_pipeline = Pipeline([
                            ("outlier",OutlierHandler(handle="winsorize")),
                            ("scale",StandardScaler())
])
                        

preprcessor=ColumnTransformer([
                            ("num",num_pipeline,num_col) ], verbose_feature_names_out=True).set_output(transform='pandas')


In [12]:
X_train_pre=preprcessor.fit_transform(X_train)
X_test_pre=preprcessor.transform(X_test)

In [13]:
X_train_pre

,num__credit_amount,num__business_age_months,num__monthly_income_avg,num__total_deposits_3m,num__revenue_volatility_3m,num__request_ratio,num__dti_monthly,num__nsf_count_3m,num__negative_days_3m,num__owner_percentage,num__owner_credit_score,num__credit_income_ratio,num__deposit_coverage,num__nsf_risk_ratio,num__owner_reliability,num__request_stress_ratio
3285,0.401950,-0.219651,0.430803,0.283029,-0.447568,-0.060274,-1.034448,-0.738280,0.695557,-0.270389,0.594829,-0.047378,-0.439733,-0.828498,-0.105230,-0.634706
2931,-0.709276,-0.759549,-0.335870,-0.261608,1.044411,-0.983692,-1.277982,-0.195427,1.171559,1.416863,-0.462126,-0.995186,1.232556,-0.228343,1.171142,-0.606925
2426,-0.442965,-0.813539,-0.153597,-0.333860,0.225046,-0.678874,1.210016,-0.195427,0.695557,0.906434,-0.359840,-0.681860,0.090135,-0.350326,0.739623,-0.579435
3608,0.952535,-1.056493,-0.144403,-0.078330,0.029376,2.008452,0.301699,-1.281134,1.171559,-2.458617,-0.393935,2.074747,-1.311964,-0.972367,-2.376695,0.560284
4658,0.386016,-0.111671,-0.560322,-0.741866,0.763137,2.243789,-0.685602,-0.738280,0.219556,-0.761913,1.310831,2.316566,-1.670433,-0.444044,-0.446720,1.430277
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4818,-0.460206,-0.327630,-0.343889,-0.264703,0.445174,-0.423365,1.032302,-0.195427,-1.684451,-0.865889,-0.496221,-0.418636,0.287243,-0.222238,-0.928421,-0.328552
2989,-0.738976,-0.813539,-1.017181,-0.937449,2.230657,0.652462,-0.007655,-0.195427,0.695557,0.901708,-0.803079,0.683156,-0.595053,0.783074,0.577762,1.803844
966,-0.721749,-0.516595,-0.524002,-0.447796,0.591926,-0.728183,0.913826,0.347426,-0.732448,-0.587044,-0.428031,-0.733057,0.760946,0.342037,-0.658010,-0.324401
2079,-0.832109,1.130094,-0.574948,-0.666311,0.102752,-0.929901,-0.330173,-1.281134,1.647561,-0.728829,-0.104125,-0.939052,0.597724,-0.857049,-0.718171,-0.390611


In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train_pre, y_train)

print("RF Train:", rf.score(X_train_pre, y_train))
print("RF Test:", rf.score(X_test_pre, y_test))

RF Train: 0.958
RF Test: 0.956


In [15]:
rf_real = RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_leaf=5, random_state=42)
rf_real.fit(X_train_pre, y_train)

print("Real RF Train:", rf_real.score(X_train_pre, y_train))
print("Real RF Test:", rf_real.score(X_test_pre, y_test))

Real RF Train: 0.96725
Real RF Test: 0.96
